# Optical Holography Simulation & Wavefront Reconstruction Recipe

This recipe combines 3 `algebrax` tools to simulate Optical Holography and Wavefront Reconstruction:

1. **Interference Recording & Holographic Reconstruction**:
   Superimposes Object Wave $O(x)$ with Reference Plane Wave $R(x)$ to record $I(x) = |O + R|^2$, and reconstructs $R(x) \cdot I(x) \propto O(x)$.
2. **Diffraction Angular Spectrum** (`algebrax.transforms.dft` & `idft`):
   Propagates reconstructed optical wavefronts across spatial frequency space using Discrete Fourier Transforms.
3. **Fringe Visibility & Information Entropy Audit** (`algebrax.probability.entropy`):
   Evaluates Michelson fringe contrast $V = \frac{I_{\max} - I_{\min}}{I_{\max} + I_{\min}}$ and Shannon information entropy $H(I)$.

In [ ]:
import cmath

from algebrax.probability import entropy
from algebrax.transforms import dft, idft

## 1. Recording Hologram Interference Pattern

We record interference intensity $I(x) = |O(x) + R(x)|^2$.

In [ ]:
object_wave = {0: 0.0, 1: 0.0, 2: 1.0 + 0.0j, 3: 0.0, 4: 0.0, 5: 0.8 + 0.6j, 6: 0.0, 7: 0.0}
reference_wave = {x: 1.0 * cmath.exp(1j * 0.25 * cmath.pi * x) for x in range(8)}

hologram_intensity = {x: abs(object_wave[x] + reference_wave[x]) ** 2 for x in range(8)}
print('Recorded Hologram Intensity Pattern I(x):', hologram_intensity)

## 2. Holographic Reconstruction via Reference Illumination

We illuminate the recorded hologram with reference beam $R(x)$ to extract reconstructed virtual image $O(x)$.

In [ ]:
reconstructed_wavefront = {}
for x in range(8):
    illuminated = reference_wave[x] * hologram_intensity[x]
    reconstructed_wavefront[x] = illuminated * reference_wave[x].conjugate() / (abs(reference_wave[x]) ** 2)

print('Reconstructed Optical Field Wavefront Amplitudes:')
for x in range(8):
    print(f'  x={x}: Amp = {abs(reconstructed_wavefront[x]):.4f}')

## 3. Diffraction Propagation via Discrete Fourier Transform (dft & idft)

We propagate the optical wavefront across angular frequencies using `dft` and `idft`.

In [ ]:
real_hologram = {x: float(I_val) for x, I_val in hologram_intensity.items()}
angular_spectrum = dft(real_hologram, n=8)
spatial_reconstruction = idft(angular_spectrum, n=8)

print('Angular Frequency Spectrum dft(I):', angular_spectrum)
print('Spatial Reconstruction idft(F):   ', spatial_reconstruction)

## 4. Fringe Visibility & Shannon Entropy Audit (entropy)

We compute Michelson fringe contrast $V$ and Shannon entropy $H(P)$.

In [ ]:
total_i = sum(hologram_intensity.values())
prob_dist = {x: val / total_i for x, val in hologram_intensity.items()}

h_entropy = entropy(prob_dist)
i_max, i_min = max(hologram_intensity.values()), min(hologram_intensity.values())
v_contrast = (i_max - i_min) / (i_max + i_min)

print(f'Fringe Visibility Contrast V: {v_contrast * 100:.2f}%')
print(f'Hologram Shannon Entropy H:    {h_entropy:.4f} bits')